# MLPC 2026 — Challenge: Sound Event Detection
**Team Temporal Frameworks** — Sebastian Pichler, Laurin Siebert

Extending the segment classifiers from Task 4 to full sound event detection (SED) for recordings of arbitrary length, predicting which of the 15 classes are active and when (onset/offset).

Pipeline mirrors the provided challenge baseline, scoring uses the official segment-based macro F1 from `evaluate.py`.

Task 1 (baseline reproduction) is handled by the provided `challenge_baseline/` notebook, this notebook starts at Task 2.

### What this notebook covers

1. Setup
2. Feature & label loading
3. Data splits
4. SED inference & evaluation helpers
5. Training data
6. Task 2 — simple classifier (our XGBoost from Task 4)
7. Hidden-test submission


## 1. Setup

In [1]:
import os
import sys
import glob
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from typing import List, Dict, Tuple, Callable

# evaluate.py is located in the 'provided_baseline' subdirectory
sys.path.insert(0, os.path.join(os.getcwd(), "provided_baseline"))
from evaluate import (
    aggregate_ground_truth_annotations,
    build_segment_frame_from_intervals,
    calculate_f1_score,
)

# copied provided functions from challenge_baseline.ipynb into sed_utils.py for better organization
from sed_utils import (
    build_feature_matrix,
    get_segment_labels,
    load_all_segments,
    run_sed_inference,
    predictions_to_intervals,
    generate_predictions,
    evaluate_split,
)

rng = np.random.default_rng(seed=42)

In [2]:
PATH_TO_DATASET = "../../data/MLPC2026_challenge"

PATH_TRAIN = os.path.join(PATH_TO_DATASET, "train")
PATH_VAL   = os.path.join(PATH_TO_DATASET, "validation")
PATH_TEST  = os.path.join(PATH_TO_DATASET, "test")

# Sanity check — will raise an AssertionError if a path does not exist.
for path in [PATH_TRAIN, PATH_VAL, PATH_TEST]:
    assert os.path.isdir(path), f"Directory not found: {path}"
for path in [
    os.path.join(PATH_TRAIN, "annotations.csv"),
    os.path.join(PATH_TRAIN, "audio_features"),
    os.path.join(PATH_VAL,   "annotations.csv"),
    os.path.join(PATH_VAL,   "audio_features"),
    os.path.join(PATH_TEST,  "audio_features"),
]:
    assert os.path.exists(path), f"Path not found: {path}"

print("Dataset paths OK.")

SEGMENT_LENGTH = 1.0  # each feature vector covers a 1-second window
HOP_SIZE       = 0.5  # segments are extracted with 50% overlap

FEATURE_NAMES = [
    "zcr_mean",        "zcr_std",        "zcr_min",        "zcr_max",
    "melspect_mean",   "melspect_std",   "melspect_min",   "melspect_max",
    "mfcc_mean",       "mfcc_std",       "mfcc_min",       "mfcc_max",
    "mfcc_d_mean",     "mfcc_d_std",     "mfcc_d_min",     "mfcc_d_max",
    "mfcc_d2_mean",    "mfcc_d2_std",    "mfcc_d2_min",    "mfcc_d2_max",
    "flux_mean",       "flux_std",       "flux_min",       "flux_max",
    "flatness_mean",   "flatness_std",   "flatness_min",   "flatness_max",
    "centroid_mean",   "centroid_std",   "centroid_min",   "centroid_max",
    "bandwidth_mean",  "bandwidth_std",  "bandwidth_min",  "bandwidth_max",
    "contrast_mean",   "contrast_std",   "contrast_min",   "contrast_max",
    "rolloff_low_mean",  "rolloff_low_std",  "rolloff_low_min",  "rolloff_low_max",
    "rolloff_high_mean", "rolloff_high_std", "rolloff_high_min", "rolloff_high_max",
    "energy_mean",     "energy_std",     "energy_min",     "energy_max",
    "power_mean",      "power_std",      "power_min",      "power_max",
]

# subset settled on in Task 4 (dropped mel-spectrogram, keep zcr / mfcc(+d, +d2) / spectral contrast / power)
# 420 dims for XGBoost
FEATURE_SELECT = [
    "zcr_mean", "zcr_std", "zcr_min", "zcr_max",
    "mfcc_mean", "mfcc_std", "mfcc_min", "mfcc_max",
    "mfcc_d_mean", "mfcc_d_std", "mfcc_d_min", "mfcc_d_max",
    "mfcc_d2_mean", "mfcc_d2_std", "mfcc_d2_min", "mfcc_d2_max",
    "contrast_mean", "contrast_std", "contrast_min", "contrast_max",
    "power_mean", "power_std", "power_min", "power_max",
]

# The 15 target sound event classes — sorted alphabetically to match the .npz annotation order.
CLASS_NAMES = [
    "bell_ringing",
    "coffee_machine",
    "cutlery_dishes",
    "door_open_close",
    "footsteps",
    "keyboard_typing",
    "keychain",
    "light_switch",
    "microwave",
    "phone_ringing",
    "running_water",
    "toilet_flushing",
    "vacuum_cleaner",
    "wardrobe_drawer_open_close",
    "window_open_close",
]

Dataset paths OK.


## 2. Feature & label loading

Each `.npz` holds per-frame features and an `annotations` array of shape `(T, C, A)` — overlap fraction per frame, class, annotator. We binarise a class as active if any annotator overlapped the frame (`> 0`) and then majority-vote across annotators (same rule as the baseline).

In [3]:
# precompute column indices of the FEATURE_SELECT subset
_sample = dict(np.load(sorted(glob.glob(os.path.join(PATH_TRAIN, "audio_features", "*.npz")))[0],allow_pickle=True))
FEATURE_WIDTH = {n: (_sample[n].shape[1] if _sample[n].ndim > 1 else 1) for n in FEATURE_NAMES}

def feature_columns(feature_names) -> np.ndarray:
    span, c = {}, 0
    for n in FEATURE_NAMES:
        span[n] = (c, c + FEATURE_WIDTH[n]); c += FEATURE_WIDTH[n]
    idx = []
    for n in feature_names:
        idx += list(range(*span[n]))
    return np.array(idx)

SELECT_COLS = feature_columns(FEATURE_SELECT)   # column indices of the 420-dim subset

## 3. Data splits

`train` / `validation` / `test` are predefined. The hidden `test` split has no labels, so — as in the baseline — we split the 999 validation recordings 50/50 into a local validation set (modelselection / tuning) and a non-hidden test set (one-shot final check, not used for tuning).

In [4]:
train_files = sorted(glob.glob(os.path.join(PATH_TRAIN, "audio_features", "*.npz")))
val_files   = sorted(glob.glob(os.path.join(PATH_VAL,   "audio_features", "*.npz")))
test_files  = sorted(glob.glob(os.path.join(PATH_TEST,  "audio_features", "*.npz")))

val_shuffled = rng.permutation(val_files).tolist()
half = len(val_shuffled) // 2
our_val_files  = val_shuffled[:half]   # local validation set
our_test_files = val_shuffled[half:]   # non-hidden local test set

print(f"train:           {len(train_files)}")
print(f"local val:       {len(our_val_files)}")
print(f"non-hidden test: {len(our_test_files)}")
print(f"hidden test:     {len(test_files)}")

# validation-split ground truth, shared by both local evaluations
ann_df = pd.read_csv(os.path.join(PATH_VAL, "annotations.csv"))

train:           3704
local val:       499
non-hidden test: 500
hidden test:     1007


## 4. SED inference & evaluation helpers

Inference per recording: predict every overlapping frame, then keep only the whole-second frames (one prediction per second, matching the 1 s metric resolution) and merge consecutive active seconds per class into onset/offset intervals.

A predict function maps a `(N, 960)` feature matrix to `(N, 15)` binary predictions, so the same inference/eval code drives both the decision-tree baseline and XGBoost.

In [5]:
#PredictFn = Callable[[np.ndarray], np.ndarray]   # (N, 960) -> (N, 15) binary

def show_results(macro_f1, results, title):
    print(f"=== {title} — macro F1: {macro_f1:.4f} ===")
    print(results[["annotation", "precision", "recall", "f1"]].to_string(index=False))

## 5. Training data

Stack all 1 s frames from the training split, treating every frame as an independent example. The XGBoost models in Task 2 train on the `FEATURE_SELECT` subset of these (sliced via `SELECT_COLS`).

In [6]:
# Load training segments from all (or a subset of) training recordings.
#
# MAX_TRAINING_FILES    — None = use all ~3 700 training files.
#                         Set to e.g. 500 for a quick test run.
# MAX_TRAINING_SEGMENTS — After loading, randomly subsample to this many segments.
#                         Keeps training fast even when loading all files.
#                         None = keep all loaded segments (can be slow to train).
#
# Tip: A decision tree does not need hundreds of thousands of examples to converge.
# 50 000 segments drawn from diverse recordings is a good default starting point.
MAX_TRAINING_FILES    = None    # None = use all training files
MAX_TRAINING_SEGMENTS = 100_000  # None = keep all loaded segments

train_files_to_use = train_files
if MAX_TRAINING_FILES is not None:
    train_files_to_use = rng.choice(
        train_files, size=min(MAX_TRAINING_FILES, len(train_files)), replace=False
    ).tolist()

print(f"Loading features from {len(train_files_to_use)} training recordings...")
X_train, Y_train = load_all_segments(train_files_to_use)
print(f"Loaded {X_train.shape[0]} segments with {X_train.shape[1]} features each.")

if MAX_TRAINING_SEGMENTS is not None and X_train.shape[0] > MAX_TRAINING_SEGMENTS:
    idx = rng.choice(X_train.shape[0], size=MAX_TRAINING_SEGMENTS, replace=False)
    X_train = X_train[idx]
    Y_train = Y_train[idx]
    print(f"Subsampled to {X_train.shape[0]} segments for faster training.")

print(f"\nFinal training set: {X_train.shape[0]} segments × {X_train.shape[1]} features, {Y_train.shape[1]} classes.")
print()
print("Class distribution — fraction of segments where class is active:")
for i, cls in enumerate(CLASS_NAMES):
    frac = Y_train[:, i].mean()
    bar  = "#" * int(frac * 40)
    print(f"  {cls:<35} {frac:.3f}  {bar}")

Loading features from 3704 training recordings...
Loaded 170508 segments with 960 features each.
Subsampled to 100000 segments for faster training.

Final training set: 100000 segments × 960 features, 15 classes.

Class distribution — fraction of segments where class is active:
  bell_ringing                        0.016  
  coffee_machine                      0.038  #
  cutlery_dishes                      0.080  ###
  door_open_close                     0.056  ##
  footsteps                           0.146  #####
  keyboard_typing                     0.103  ####
  keychain                            0.058  ##
  light_switch                        0.016  
  microwave                           0.076  ###
  phone_ringing                       0.075  ##
  running_water                       0.136  #####
  toilet_flushing                     0.034  #
  vacuum_cleaner                      0.066  ##
  wardrobe_drawer_open_close          0.031  #
  window_open_close                   0.020  


## 6. Task 2 — Simple classifier: XGBoost from Task 4

**2(a).** Our best classical model from Task 4 was a per-class binary XGBoost (binary relevance — one model per class). Gradient-boosted trees were robust on the tabular features with little preprocessing, key modification was a per-class `scale_pos_weight = neg/pos` to counter the heavy frame-level class imbalance (slightly improved rare classes like `light_switch`). 

Core hyperparameters: `n_estimators=100`, `learning_rate=0.1`, `max_depth=6`, `min_child_weight=3`, `subsample=0.8`, `tree_method="hist"`. Features: Task 4 subset (`FEATURE_SELECT`, 420 dims).

> Our overall best in Task 4 was actually the focal-loss MLP, but per task instruction we use our best classical model here.

In [13]:
def select_features(X):
    """Full (N, 960) feature matrix -> (N, 420) FEATURE_SELECT subset."""
    return X[:, SELECT_COLS]

def train_xgb_per_class(X, Y, **params):
    """Binary-relevance XGBoost: one model per class, per-class positive weighting."""
    Xs = select_features(X)
    models = {}
    for ci, cls in enumerate(CLASS_NAMES):
        y = Y[:, ci]
        pos = int(np.sum(y == 1))
        neg = int(np.sum(y == 0))
        models[cls] = XGBClassifier(
            objective="binary:logistic",
            eval_metric="aucpr",
            tree_method="hist",
            scale_pos_weight=(neg / pos if pos else 1.0),
            n_jobs=-1,
            random_state=42,
            **params,
        ).fit(Xs, y)
    return models

def xgb_predict_fn(models):
    """Wrap per-class models into a (N, 960) -> (N, 15) predictor (slices FEATURE_SELECT)."""
    def predict(X):
        Xs = select_features(X)
        return np.column_stack([models[c].predict(Xs) for c in CLASS_NAMES])
    return predict

# paramters from Task 4
XGB_PARAMS = dict(n_estimators=100, learning_rate=0.1, max_depth=6, min_child_weight=3, subsample=0.8)

print("training per-class XGBoost...")
t = time.time()
xgb_models = train_xgb_per_class(X_train, Y_train, **XGB_PARAMS)
print(f"trained {len(xgb_models)} models in {time.time() - t:.1f}s")

xgb_predict = xgb_predict_fn(xgb_models)
pred_val_xgb  = generate_predictions(our_val_files,  xgb_predict)
pred_test_xgb = generate_predictions(our_test_files, xgb_predict)

f1_val_xgb,  res_val_xgb  = evaluate_split(pred_val_xgb,  our_val_files,  ann_df)
f1_test_xgb, res_test_xgb = evaluate_split(pred_test_xgb, our_test_files, ann_df)

show_results(f1_val_xgb,  res_val_xgb,  "XGBoost / local val")
print()
show_results(f1_test_xgb, res_test_xgb, "XGBoost / non-hidden test")

training per-class XGBoost...
trained 15 models in 61.0s
=== XGBoost / local val — macro F1: 0.5045 ===
                annotation  precision   recall       f1
              bell_ringing   0.324786 0.387755 0.353488
            coffee_machine   0.383199 0.639155 0.479137
            cutlery_dishes   0.461739 0.744128 0.569868
           door_open_close   0.242674 0.620609 0.348914
                 footsteps   0.463975 0.708729 0.560811
           keyboard_typing   0.596833 0.718475 0.652029
                  keychain   0.414886 0.682424 0.516040
              light_switch   0.381107 0.546729 0.449136
                 microwave   0.463335 0.680648 0.551351
             phone_ringing   0.537897 0.673469 0.598097
             running_water   0.769299 0.812675 0.790392
           toilet_flushing   0.380645 0.710843 0.495798
            vacuum_cleaner   0.609731 0.789544 0.688084
wardrobe_drawer_open_close   0.196242 0.503571 0.282424
         window_open_close   0.148259 0.530249 0.231726
